# Overlap Verification
Two things:
1. Read the 18 flagged ILDC cases from parquet so you can visually confirm they match the PredEx case
2. TF-IDF cosine similarity as a more robust second check

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
ILDC_TRAIN_PATH = "/content/drive/MyDrive/CAPSTONE/Data/preprocessed_ildc/criminal_train.parquet"
ILDC_DEV_PATH   = "/content/drive/MyDrive/CAPSTONE/Data/preprocessed_ildc/criminal_dev.parquet"
PREDEX_PATH     = "/content/drive/MyDrive/CAPSTONE/Data/predex_test/predex_test_final.csv"
REPORT_PATH     = "/content/drive/MyDrive/CAPSTONE/Data/predex_test/overlap_report.csv"

In [7]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## 1. Read the 18 flagged ILDC cases and display side-by-side with PredEx

In [8]:
report    = pd.read_csv(REPORT_PATH)
flagged_ildc_ids = report['ildc_id'].tolist()

ildc_train = pd.read_parquet(ILDC_TRAIN_PATH)
ildc_dev   = pd.read_parquet(ILDC_DEV_PATH)
ildc_all   = pd.concat([ildc_train, ildc_dev], ignore_index=True)

predex = pd.read_csv(PREDEX_PATH)

# Pull only the 18 flagged ILDC rows
flagged_ildc = ildc_all[ildc_all['id'].isin(flagged_ildc_ids)].set_index('id')

print(f"Flagged ILDC cases found in parquet: {len(flagged_ildc)} / {len(flagged_ildc_ids)}")

Flagged ILDC cases found in parquet: 18 / 18


In [9]:
# Display first 800 chars of each pair so you can visually confirm
pd.set_option('display.max_colwidth', 800)

rows = []
for _, r in report.iterrows():
    ildc_id   = r['ildc_id']
    case_name = r['predex_case_name']

    ildc_text   = flagged_ildc.loc[ildc_id, 'text'][:800] if ildc_id in flagged_ildc.index else 'NOT FOUND'
    predex_text = predex[predex['Case Name'] == case_name]['Input'].values
    predex_text = predex_text[0][:800] if len(predex_text) > 0 else 'NOT FOUND'

    rows.append({
        'predex_case_name' : case_name,
        'ildc_id'          : ildc_id,
        'ildc_text_start'  : ildc_text,
        'predex_text_start': predex_text
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,predex_case_name,ildc_id,ildc_text_start,predex_text_start
0,State of U.P Vs. Vad Narain,1992_315,"Ratnavel Pandian, J. The State of U.P. on being dissatisfied with the judgment of the High Court of Allahabad rendered in Criminal Appeals Nos. 253 and 256 of 1979, setting aside the companyviction of the appellant under Section 302 read with Section 34, I.P.C. and the sentence of death imposed by the Trial Court, has preferred this appeal. It may also be pointed out in this companynection that the reference under Section 366 of the CrPC made by the learned Sessions Judge which was numbered as Capital Sentence Case No. 7 of 1979 for the companyfirmation of the death sentence under Section 366 of the CrPC has been rejected by the High Court. The respondent Vad Narain and 8 others were charged on the allegation that on 24th March 1978 at about 9-10 P.M. in the village called Saurmau P. ...","S. RATNAVEL PANDIAN, J. 1. The State of U.P. on being dissatisfied with the judgment of the High Court of Allahabad rendered in Criminal Appeal Nos. 253 and 256 of 1979, setting aside the conviction of the appellant under Section 302 read with Section 34 IPC and the sentence of death imposed by the trial court, has preferred this appeal. It may also be pointed out in this connection that the reference under Section 366 of the Code of Criminal Procedure made by the learned Sessions Judge which was numbered as Capital Sentence Case No. 7 of 1979 for the confirmation of the death sentence under Section 366 of the Code of Criminal Procedure has been rejected by the High Court. 2. The respondent Ved Narain and 8 others were charged on the allegation that on March 24, 1978 at about 9.10 p.m...."
1,Shivaji @ Dadya Shankar Alhat Vs. State Of Maharashtra,2004_719,"Arising out of SLP Crl. No. 4982 of 2003 ARIJIT PASAYAT, J Leave granted. The State of U.P. is in appeal against the judgment of the learned Single Judge of the Allahabad High Court, Lucknow Bench. By the said impugned judgment, Criminal Appeal No.37 of 1995 was disposed of by reducing the respondents sentence of 7 years RI imposed in respect of offence punishable under Section 304 Part II of the Indian Penal Code, 1860 in short the IPC to the period already undergone with a direction to pay fine of Rs.15,000/- with default stipulation of one year RI. The respondent hereafter referred to as the accused was found guilty by the learned Sessions Judge, Sitapur. The accused had faced trial for offence punishable under Section 302 IPC for having caused homicidal death of one Chetai...","Dr. Arijit Pasayat, J. 1. Leave granted. A large number of cases in recent times coming before this Court involving rape and murder of young girls, is a matter of concern. In the instant case victim was about nine years of age who was the victim of sexual assault and animal lust of the accused-appellant; she was not only raped but was murdered by the accused appellant. 2. Learned Second Additional Judge, Pune in Sessions Case No.209 of 2002 tried the appellant for offences punishable under Sections 302 and Section 376 (2)(f) of the Indian Penal Code, 1860 (in short the ‘IPC). By judgment and order dated 27th June, 2004, the trial court found the appellant guilty for the aforesaid offences and he was sentenced to death for the offence of murder and in respect of the other offence senten..."
2,Bhagwan Singh Vs. State Of Punjab,1992_654,"CRIMINAL APPELLATE JURISDICTION Criminal Appeal Nos. 388 and 666 of 1981. From the Judgment and Order dated 17.2.1981 of the Punjab Haryana High Court in Crl. A. Nos. 1392 and 956 of 1979. L. Kohli, R.C. Kohli and M.C. Dhingra N.P. for the Appellants. M. Singh and R.S. Suri for the Respondent. The Judgment of the Court was delivered by, JAYACHANDRA REDDY, J. These two appeals are filed against a companymon judgment of the Punjab Haryana High Court in Criminal Appeal Nos. 956/79, 1392/79 and 1310/79. There were four appellants in Criminal Appeal No. 956/79 before the High Court namely 

## 2. TF-IDF cosine similarity verification

For each of the 18 flagged pairs, compute cosine similarity between ILDC text and PredEx Input.
A score > 0.85 strongly confirms they are the same case (even with OCR noise).
Also checks a sample of non-flagged PredEx cases to confirm they score low.

In [10]:
def clean(text):
    return re.sub(r'[^a-z0-9 ]', '', str(text).lower())

# Build corpus: 18 ILDC flagged texts + their PredEx counterparts
ildc_texts   = []
predex_texts = []
pair_labels  = []

for _, r in report.iterrows():
    ildc_id   = r['ildc_id']
    case_name = r['predex_case_name']

    if ildc_id not in flagged_ildc.index:
        continue

    predex_row = predex[predex['Case Name'] == case_name]['Input'].values
    if len(predex_row) == 0:
        continue

    ildc_texts.append(clean(flagged_ildc.loc[ildc_id, 'text']))
    predex_texts.append(clean(predex_row[0]))
    pair_labels.append((case_name, ildc_id))

print(f"Pairs ready for TF-IDF check: {len(pair_labels)}")

Pairs ready for TF-IDF check: 18


In [11]:
# Fit TF-IDF on all texts together
all_texts = ildc_texts + predex_texts
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(all_texts)

n = len(pair_labels)
ildc_vecs   = tfidf_matrix[:n]
predex_vecs = tfidf_matrix[n:]

# Pairwise similarity for each matched pair
scores = []
for i in range(n):
    sim = cosine_similarity(ildc_vecs[i], predex_vecs[i])[0][0]
    scores.append(sim)

results_df = pd.DataFrame({
    'predex_case_name': [p[0] for p in pair_labels],
    'ildc_id'         : [p[1] for p in pair_labels],
    'cosine_sim'      : scores,
    'verdict'         : ['CONFIRMED OVERLAP' if s > 0.7 else 'REVIEW MANUALLY' for s in scores]
})

results_df = results_df.sort_values('cosine_sim', ascending=False)
print(results_df.to_string(index=False))

                                                                                                                          predex_case_name   ildc_id  cosine_sim           verdict
                                                                                                       Sonali Mukherjee Vs. Union Of India 2009_1889    0.983464 CONFIRMED OVERLAP
                                                                                 State Of Andhra Pradesh Vs. Dr. M.V. Ramana Reddy And Ors  1991_237    0.981458 CONFIRMED OVERLAP
                                                                                                        Bhagwan Singh  Vs. State Of Punjab  1992_654    0.979552 CONFIRMED OVERLAP
                                                                                             Kahan Singh &amp; Others Vs. State of Haryana  1971_111    0.974894 CONFIRMED OVERLAP
                                                                                                 State of

In [12]:
# Sanity check: pick 20 random NON-flagged PredEx cases
# and compute their max similarity against all 18 ILDC flagged texts
# These should all score LOW (< 0.5)

non_flagged = predex[~predex['Case Name'].isin(report['predex_case_name'])].sample(20, random_state=42)
non_flagged_texts = [clean(t) for t in non_flagged['Input'].tolist()]

nf_vecs = vectorizer.transform(non_flagged_texts)

# Max similarity of each non-flagged case against any of the 18 ILDC cases
sim_matrix = cosine_similarity(nf_vecs, ildc_vecs)  # (20, 18)
max_sims   = sim_matrix.max(axis=1)

sanity_df = pd.DataFrame({
    'predex_case_name': non_flagged['Case Name'].values,
    'max_sim_vs_ildc' : max_sims,
    'verdict'         : ['SUSPICIOUS - CHECK' if s > 0.7 else 'OK (not in ILDC)' for s in max_sims]
})

print(sanity_df.to_string(index=False))

                                                                        predex_case_name  max_sim_vs_ildc            verdict
                                             Saroj &amp; Others Vs. Het Lal &amp; Others         0.642647   OK (not in ILDC)
                                                         State Of H.P Vs. Narender Kumar         0.730416 SUSPICIOUS - CHECK
                     Rakesh Baban Borhade Vs. The State Of Maharashtra &amp; Anr Etc.Etc         0.679158   OK (not in ILDC)
                                    P.RAJAGOPAL AND ORS. ETC Vs. THE STATE OF TAMIL NADU         0.828973 SUSPICIOUS - CHECK
                   THE STATE OF ARUNACHAL PRADESH Vs. RAMCHANDRA RABIDAS @ RATAN RABIDAS         0.717093 SUSPICIOUS - CHECK
                                              Gyanendra Kumar Vs. State of Uttar Pradesh         0.672226   OK (not in ILDC)
                             Cref Finance Ltd Vs. Shree Shanthi Homes Pvt. Ltd. &amp;Anr         0.677910   OK (not in ILDC)


In [13]:
# Summary
confirmed  = (results_df['verdict'] == 'CONFIRMED OVERLAP').sum()
needs_review = (results_df['verdict'] == 'REVIEW MANUALLY').sum()
suspicious = (sanity_df['verdict'] == 'SUSPICIOUS - CHECK').sum()

print("=== SUMMARY ===")
print(f"Flagged pairs confirmed as overlaps (cosine > 0.7) : {confirmed} / {n}")
print(f"Flagged pairs that need manual review               : {needs_review}")
print(f"Non-flagged cases suspiciously similar to ILDC      : {suspicious} / 20")
if suspicious > 0:
    print("  --> Window method may have missed some overlaps. Re-run with lower threshold.")
else:
    print("  --> Non-flagged cases look clean. Window method is reliable.")

=== SUMMARY ===
Flagged pairs confirmed as overlaps (cosine > 0.7) : 15 / 18
Flagged pairs that need manual review               : 3
Non-flagged cases suspiciously similar to ILDC      : 10 / 20
  --> Window method may have missed some overlaps. Re-run with lower threshold.
